In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
tf.__version__

2025-12-30 13:40:41.820659: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767102041.973648      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767102042.018804      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

'2.18.0'

In [2]:
batch_size = 64
split = 0.2
epochs = 5

In [3]:
class celeb_dataframe():
    '''Wraps the celebA dataset, allowing an easy way to:
       - Select the features of interest,
       - Split the dataset into 'training', 'test' or 'validation' partition.
    '''
    def __init__(self,main_folder = "/kaggle/input/celeba-dataset" , selected_features = [] , drop_features = []):
        self.main_folder = main_folder
        self.images_folder   = os.path.join(main_folder, 'img_align_celeba/img_align_celeba')
        self.attr_path = os.path.join(main_folder , "list_attr_celeba.csv")
        self.features_name = []
        self.prepare(drop_features , selected_features)
        
    def prepare(self,drop_features, selected_features ):
        
        #attributes selection
        if len(selected_features) == 0  :
            self.attributes = pd.read_csv(self.attr_path)
            self.num_features = 40
        else:
            self.num_features = len(selected_features)
            selected_features = selected_features.copy()
            selected_features.append("image_id")
            self.attributes = pd.read_csv(self.attr_path)[selected_features]
        
        #removing features
        if len(drop_features) != 0:
            for feature in drop_features:
                if feature in self.attributes:
                    self.attributes = self.attributes.drop(feature , axis = 1)
                    self.num_features -= 1
            
        self.attributes.set_index("image_id" , inplace = True)
        self.attributes.replace(to_replace = -1 , value = 0 , inplace = True)
        self.attributes["image_id"] = list(self.attributes.index)
        self.features_name = list(self.attributes.columns)[:-1]
        
        return self.attributes

In [4]:
features  = [
    'Male',
    'Bald', 'Bangs', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Gray_Hair',
    'Straight_Hair', 'Wavy_Hair', 'Receding_Hairline',
    'Mustache', 'Sideburns', 'No_Beard',
    'Eyeglasses', 'Wearing_Hat', 'Wearing_Earrings',
    'Wearing_Necklace', 'Wearing_Necktie'
]

In [5]:
celeb_df = celeb_dataframe()
celeb = celeb_df.prepare(drop_features=[],selected_features= features) #taking all the features for training.
celeb.head(10)


,Male,Bald,Bangs,Black_Hair,Blond_Hair,Brown_Hair,Gray_Hair,Straight_Hair,Wavy_Hair,Receding_Hairline,Mustache,Sideburns,No_Beard,Eyeglasses,Wearing_Hat,Wearing_Earrings,Wearing_Necklace,Wearing_Necktie,image_id
image_id,,,,,,,,,,,,,,,,,,,
000001.jpg,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,000001.jpg
000002.jpg,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,000002.jpg
000003.jpg,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,000003.jpg
000004.jpg,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,1,0,000004.jpg
000005.jpg,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,000005.jpg
000006.jpg,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,000006.jpg
000007.jpg,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,000007.jpg
000008.jpg,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,000008.jpg
000009.jpg,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,000009.jpg


In [6]:
print("Number of features:",len(celeb_df.features_name))

Number of features: 18


In [7]:
split_ind = int((1 - split) * celeb.shape[0])
train_df = celeb[:split_ind]
validation_df = celeb[split_ind:]

In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# augumentations for training set:
train_datagen = ImageDataGenerator(rotation_range=20, 
                                   rescale=1./255, 
                                   width_shift_range=0.2, 
                                   height_shift_range=0.2, 
                                   shear_range=0.2, 
                                   zoom_range=0.2, 
                                   horizontal_flip=True, 
                                   fill_mode='nearest')

valid_datagen = ImageDataGenerator(rescale= 1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=celeb_df.images_folder,
    x_col='image_id',
    y_col=celeb_df.features_name,
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='raw',
    shuffle = True
)

validation_generator = valid_datagen.flow_from_dataframe(dataframe=validation_df,
                                                         directory=celeb_df.images_folder,
                                                         x_col='image_id',
                                                         y_col=celeb_df.features_name,
                                                         target_size=(224, 224),
                                                         batch_size=batch_size,
                                                         class_mode='raw'
                                                        )                


Found 162079 validated image filenames.
Found 40520 validated image filenames.


In [9]:
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
import tensorflow as tf

def classifier(num_features):
    base = MobileNetV3Large(
        input_shape=(224,224,3),
        weights=None,
        include_top=False,
        pooling="avg"
    )

    x = base.output
    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    output = Dense(num_features, activation='sigmoid')(x)

    model = tf.keras.models.Model(inputs=base.input, outputs=output)
    return model


In [10]:
num_features = len(celeb_df.features_name)
cls = classifier(num_features)
cls.summary()

I0000 00:00:1767102639.359606      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 112, 112,  │        432 │ rescaling[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 112, 112,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 112, 112,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        144 │ activation[0][0]  │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        256 │ re_lu[0][0]       │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_add   │ (None, 112, 112,  │          0 │ activation[0][0], │
│ (Add)               │ 16)               │            │ expanded_conv_pr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 112, 112,  │      1,024 │ expanded_conv_ad… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 112, 112,  │        256 │ expanded_conv_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 112, 112,  │          0 │ expanded_conv_1_… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 113, 113,  │          0 │ re_lu_1[0][0]     │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 56, 56,    │        576 │ expanded_conv_1_… │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 56, 56,    │        256 │ expanded_conv_1_

 Total params: 4,002,962 (15.27 MB)

 Trainable params: 3,976,514 (15.17 MB)

 Non-trainable params: 26,448 (103.31 KB)

In [12]:
cls.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(0.001),
    metrics=['binary_accuracy']
)


In [14]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

os.makedirs("/kaggle/working/ckpts/", exist_ok=True)

earlystop = EarlyStopping(
    monitor="val_binary_accuracy",
    patience=3,
    restore_best_weights=True
)

checkpoint_filepath = (
    "/kaggle/working/ckpts/"
    f"weights-FC{celeb_df.num_features}-MobileNetV3"
    "-{val_binary_accuracy:.2f}.weights.h5"
)

model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_binary_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)


In [16]:
history = cls.fit(
    train_generator,
    epochs=epochs,
    steps_per_epoch=len(train_generator),
    validation_data=validation_generator,
    validation_steps=len(validation_generator),
    callbacks = [earlystop , model_checkpoint],
    verbose=1)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


I0000 00:00:1767103377.036064     109 service.cc:148] XLA service 0x7f5a7800b740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767103377.036810     109 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767103381.027092     109 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1767103388.811386     109 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1767103389.001991     109 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1767103389.506145     109 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1767103389.7036

1857/2533 ━━━━━━━━━━━━━━━━━━━━ 13:05 1s/step - binary_accuracy: 0.8509 - loss: 0.3730

E0000 00:00:1767105571.051799     109 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1767105571.236060     109 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1767105571.670927     109 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1767105571.868433     109 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


2533/2533 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - binary_accuracy: 0.8584 - loss: 0.3542
Epoch 1: val_binary_accuracy improved from -inf to 0.85583, saving model to /kaggle/working/ckpts/weights-FC18-MobileNetV3-0.86.weights.h5
2533/2533 ━━━━━━━━━━━━━━━━━━━━ 3386s 1s/step - binary_accuracy: 0.8584 - loss: 0.3542 - val_binary_accuracy: 0.8558 - val_loss: 0.4211
Epoch 2/5
2533/2533 ━━━━━━━━━━━━━━━━━━━━ 0s 795ms/step - binary_accuracy: 0.9076 - loss: 0.2175
Epoch 2: val_binary_accuracy did not improve from 0.85583
2533/2533 ━━━━━━━━━━━━━━━━━━━━ 2141s 845ms/step - binary_accuracy: 0.9076 - loss: 0.2175 - val_binary_accuracy: 0.8454 - val_loss: 0.6108
Epoch 3/5
2533/2533 ━━━━━━━━━━━━━━━━━━━━ 0s 785ms/step - binary_accuracy: 0.9198 - loss: 0.1874
Epoch 3: val_binary_accuracy did not improve from 0.85583
2533/2533 ━━━━━━━━━━━━━━━━━━━━ 2117s 836ms/step - binary_accuracy: 0.9198 - loss: 0.1874 - val_binary_accuracy: 0.6870 - val_loss: 12.0911
Epoch 4/5
2533/2533 ━━━━━━━━━━━━━━━━━━━━ 0s 781ms/step - b

In [19]:
save_dir = "/kaggle/working/models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(
    save_dir, 
    "My_celebA_attr_Classifier_model.keras"
)

cls.save(model_path)


In [21]:
weights_dir = "/kaggle/working/models/weights"
os.makedirs(weights_dir, exist_ok=True)

weights_path = os.path.join(
    weights_dir,
    "celebA_mobilenetv3.weights.h5"
)

cls.save_weights(weights_path)